In [1]:
import pysftp
import sys
import os
import pandas as pd

In [2]:
#download files again (yes/no)?
download = "yes"

#set year for data creation
year = '2020'

In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
password = "G2JQwhAF6KmyjVs."                
username = "jonas.savelsberg@unibas.ch"                
port = '22'

In [4]:
dir_out = "../parsed_data/"

In [5]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [6]:
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")

    # show list of files
    files = sftp.listdir('/TP_export/')   
    #print(files)

Connection succesfully established.


## load data

In [7]:
#set paths and get file names
path_load = path+'ActualTotalLoad/'
path_load_local = path_local+'load/'
with pysftp.Connection(host=host, username=username, password=password) as sftp:
    print("Connection succesfully established.")
    # show list of files
    files = sftp.listdir(path_load)
    #download files
    if year != "":
        files = [i for i in files if year in i]

Connection succesfully established.


In [8]:
#download aggregated load data (ActualTotalLoad)
if download =="yes":
    with pysftp.Connection(host=host, username=username, password=password) as sftp:
        for file in files:
            sftp.get(path_load+file,path_load_local+file)
            print('Successfully downloaded file '+file)

Successfully downloaded file 2020_10_ActualTotalLoad.csv
Successfully downloaded file 2020_11_ActualTotalLoad.csv
Successfully downloaded file 2020_12_ActualTotalLoad.csv
Successfully downloaded file 2020_1_ActualTotalLoad.csv
Successfully downloaded file 2020_2_ActualTotalLoad.csv
Successfully downloaded file 2020_3_ActualTotalLoad.csv
Successfully downloaded file 2020_4_ActualTotalLoad.csv
Successfully downloaded file 2020_5_ActualTotalLoad.csv
Successfully downloaded file 2020_6_ActualTotalLoad.csv
Successfully downloaded file 2020_7_ActualTotalLoad.csv
Successfully downloaded file 2020_8_ActualTotalLoad.csv
Successfully downloaded file 2020_9_ActualTotalLoad.csv


In [9]:
#combine files to one data frame
df_load = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_load_local+file,
                          decimal=".",encoding="UTF-16LE",sep="\t",
                          parse_dates=True, index_col="DateTime").drop("areacode", axis=1)
    df_load = df_load.append(df_temp)
df_load = df_load[df_load.AreaTypeCode == "CTY"].drop(["AreaTypeCode",'Year','Month','Day','ResolutionCode','AreaName','UpdateTime'], axis=1).reset_index()
df_load = df_load.sort_values(by=['DateTime'])
df_load.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 487485 entries, 152584 to 97272
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DateTime        487485 non-null  datetime64[ns]
 1   MapCode         487485 non-null  object        
 2   TotalLoadValue  487485 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 14.9+ MB


In [10]:
df_load = df_load.rename(columns={"MapCode": "country", 'DateTime':'date','TotalLoadValue':'load'})
df_load = df_load.pivot_table("load", "date", "country")

In [11]:
df_load.head()

country,AT,BA,BE,BG,CH,CY,CZ,DE,DK,EE,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UA
date,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,5873.6,1386.63,8849.95,4063.0,7012.49,413.46,5549.62,42712.82,3229.98,779.1,...,11221.73,15786.89,14053.13,4972.1,5796.0,5514.0,14860.0,1167.85,2701.0,15062.0
2020-01-01 00:15:00,5803.2,NaN,8764.91,NaN,NaN,NaN,NaN,42532.17,NaN,NaN,...,11243.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 00:30:00,5737.2,NaN,8668.97,NaN,NaN,393.00,NaN,42162.57,NaN,NaN,...,11182.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 00:45:00,5658.8,NaN,8517.84,NaN,NaN,NaN,NaN,41879.33,NaN,NaN,...,11089.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-01 01:00:00,5722.0,1317.19,8457.19,3924.0,7134.25,375.05,5559.53,41694.57,3124.34,759.2,...,11060.77,15532.00,13563.78,4806.2,5621.0,5290.0,14595.0,1128.24,2658.0,14549.0


In [12]:
#some values are reported quarter hourly so we have to resample to hourly values
df_load_hourly = df_load.resample('1H').mean()

Albania is completely missing so we copy profile from ME

In [13]:
df_load_hourly['AL'] = df_load_hourly['ME']*0.0001

In [14]:
#Lets see how complete values are
df_load_hourly.count().T

country
AT    8784
BA    8651
BE    8784
BG    8784
CH    8784
CY    7699
CZ    8784
DE    8784
DK    8782
EE    8780
ES    8784
FI    8784
FR    8781
GB    8718
GR    8782
HR    8784
HU    8783
IE    8680
IT    8784
LT    8784
LU    8784
LV    8777
MD    8532
ME    8758
MK    7440
NL    8784
NO    8783
PL    8784
PT    8784
RO    8784
RS    8784
SE    8784
SI    8784
SK    8784
UA    8784
AL    8758
dtype: int64

In [15]:
#Let's see how totals behave:
df_load_hourly.groupby(lambda x: x.year).sum().T/1000000
#they are too low so we prepare Eurostat yearly for upscaling in separate sheet

,2020
country,
AT,61.057537
BA,11.148132
BE,81.146056
BG,36.503747
CH,62.419175
CY,3.658639
CZ,64.290477
DE,485.782851
DK,34.097609


In [16]:
df_load_hourly.to_csv(dir_out+'load_'+year+'_hourly_entsoe.csv', encoding="utf-8")